# Pan-UK Biobank: a multi-million-variant Manhattan plot

Each Pan-UKBB per-phenotype file contains 28,987,534 variants. This
notebook reads indexed windows from the standing-height GWAS and plots
the released `−log10(p)` statistic across all autosomes. The full
2.0 GB flat file stays remote; only its 2 MB tabix index and requested
BGZF blocks are transferred.

Increase `PANUKBB_WINDOWS` or `PANUKBB_VARIANTS_PER_WINDOW` for a
denser run.

**Source:** [Pan-UKBB downloads](https://pan.ukbb.broadinstitute.org/downloads/index.html)
and [per-phenotype file documentation](https://pan.ukbb.broadinstitute.org/docs/per-phenotype-files/index.html).
The data are CC BY 4.0; publications should acknowledge Pan-UKBB and
UK Biobank as requested on the download page.

Install beside XY with `python -m pip install numpy pysam requests xy`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pysam
import requests

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/"
    "continuous-50-both_sexes-irnt.tsv.bgz"
)
INDEX_URL = (
    "https://pan-ukb-us-east-1.s3.amazonaws.com/"
    "sumstats_flat_files_tabix/"
    "continuous-50-both_sexes-irnt.tsv.bgz.tbi"
)
index_path = DATA_DIR / "continuous-50-both_sexes-irnt.tsv.bgz.tbi"
if not index_path.exists():
    response = requests.get(INDEX_URL, timeout=120)
    response.raise_for_status()
    index_path.write_bytes(response.content)

CHROMOSOME_LENGTHS = {
    1: 249_250_621,
    2: 243_199_373,
    3: 198_022_430,
    4: 191_154_276,
    5: 180_915_260,
    6: 171_115_067,
    7: 159_138_663,
    8: 146_364_022,
    9: 141_213_431,
    10: 135_534_747,
    11: 135_006_516,
    12: 133_851_895,
    13: 115_169_878,
    14: 107_349_540,
    15: 102_531_392,
    16: 90_354_753,
    17: 81_195_210,
    18: 78_077_248,
    19: 59_128_983,
    20: 63_025_520,
    21: 48_129_895,
    22: 51_304_566,
}
windows_per_chromosome = int(os.getenv("PANUKBB_WINDOWS", "8"))
variants_per_window = int(os.getenv("PANUKBB_VARIANTS_PER_WINDOW", "15000"))
window_width = int(os.getenv("PANUKBB_WINDOW_BP", "3000000"))
if min(windows_per_chromosome, variants_per_window, window_width) <= 0:
    raise ValueError("Pan-UKBB window controls must be positive")

In [ ]:
offsets = {}
running_offset = 0
for chromosome, length in CHROMOSOME_LENGTHS.items():
    offsets[chromosome] = running_offset
    running_offset += length

position_parts = []
significance_parts = []
chromosome_parts = []

# The immutable quantitative-trait schema starts with:
# chr, pos, ref, alt, af_meta_hq, beta_meta_hq, se_meta_hq,
# neglog10_pval_meta_hq. Its plain TSV header is skipped by tabix.
position_column = 1
pvalue_column = 7

with pysam.TabixFile(DATA_URL, index=str(index_path)) as summary:
    for chromosome, length in CHROMOSOME_LENGTHS.items():
        centers = (
            (np.arange(windows_per_chromosome, dtype=np.float64) + 0.5)
            * length
            / windows_per_chromosome
        )
        starts = np.clip(
            centers - window_width / 2,
            0,
            max(0, length - window_width),
        ).astype(np.int64)
        positions = []
        significance = []
        for start in starts:
            kept = 0
            records = summary.fetch(
                str(chromosome),
                int(start),
                int(start + window_width),
            )
            for line in records:
                fields = line.split("\t")
                value = fields[pvalue_column]
                if value == "NA":
                    continue
                positions.append(offsets[chromosome] + int(fields[position_column]))
                significance.append(float(value))
                kept += 1
                if kept >= variants_per_window:
                    break

        position_parts.append(np.asarray(positions, dtype=np.float64))
        significance_parts.append(np.asarray(significance, dtype=np.float64))
        chromosome_parts.append(np.full(len(positions), chromosome, dtype=np.float64))
        print(f"chr{chromosome}: {len(positions):,} variants")

genomic_position = np.concatenate(position_parts)
neglog10_pvalue = np.concatenate(significance_parts)
chromosome_number = np.concatenate(chromosome_parts)
print(f"{genomic_position.size:,} variants total")

In [ ]:
tick_values = [
    offsets[chromosome] + CHROMOSOME_LENGTHS[chromosome] / 2 for chromosome in CHROMOSOME_LENGTHS
]
tick_labels = [str(chromosome) for chromosome in CHROMOSOME_LENGTHS]

chart = xy.scatter_chart(
    xy.scatter(
        genomic_position,
        neglog10_pvalue,
        color=chromosome_number,
        color_domain=(1, 22),
        colormap="turbo",
        size=1.2,
        opacity=0.65,
        density=True,
    ),
    xy.hline(
        -np.log10(5e-8),
        text="genome-wide significance",
        color="#fb7185",
        width=2,
        style={"dash": "6,5"},
    ),
    xy.x_axis(
        label="chromosome",
        tick_values=tick_values,
        tick_labels=tick_labels,
    ),
    xy.y_axis(label="-log10(p)"),
    xy.colorbar(title="chromosome"),
    xy.theme(
        background="#07101f",
        text_color="#e2e8f0",
        grid_color="#1e293b",
        axis_color="#94a3b8",
    ),
    title=(f"Pan-UKBB standing-height GWAS · {genomic_position.size:,} variants"),
    width=1150,
    height=620,
)
print(chart.memory_report()["canonical_bytes"], "canonical bytes")
chart